In [ ]:
import numpy as np
from typing import List, Union, Tuple
import warnings
import multiprocessing as mp
from functools import partial
import os

In [ ]:
import flammkuchen as fl
from pathlib import Path
from glob import glob
from split_dataset import SplitDataset
import time

import shutil 

In [ ]:
def no_regressor_frames(regressors, threshold=0.05):
    return np.where(np.all(regressors < threshold, axis=0))[0]

In [ ]:

def calculate_dff_with_offset_correction(
    stack: np.ndarray,
    baseline_frames,
    min_baseline_value: float = 1.0,
    per_plane_baselines: bool = False,
    offset_correction: str = 'min_subtract',
    percentile_value: float = 1.0,
    global_offset: float = None,
    timing: bool = True
) -> np.ndarray:
    """
    Calculate ΔF/F for a 4D calcium imaging stack with offset correction.
    
    Parameters
    ----------
    stack : np.ndarray
        4D array with dimensions (time, z, x, y)
    baseline_frames : list or array
        Frame indices for baseline calculation
    min_baseline_value : float, optional
        Minimum value for the baseline, by default 1.0
    per_plane_baselines : bool, optional
        Whether to use different baseline frames for each z-plane, by default False
    offset_correction : str, optional
        Method for offset correction:
        - 'min_subtract': Subtract minimum (or percentile) value
        - 'positive_shift': Shift entire dataset to ensure all values are positive
        - 'manual': Use provided global_offset value
        - 'none': No correction
        By default 'min_subtract'
    percentile_value : float, optional
        Percentile to use for 'min_subtract' method (1.0 = 1st percentile), by default 1.0
    global_offset : float, optional
        Manual offset value to add if offset_correction='manual', by default None
    timing : bool, optional
        Whether to print timing information, by default True
        
    Returns
    -------
    np.ndarray
        4D array with ΔF/F values
    """
    if timing:
        start_time = time.time()
    
    # Convert to float if needed
    if not np.issubdtype(stack.dtype, np.floating):
        stack = stack.astype(np.float32)
        if timing:
            print(f"Data type conversion: {time.time() - start_time:.2f} seconds")
    
    # Get dimensions
    t_max, z_max, x_max, y_max = stack.shape
    
    # Apply offset correction
    if timing:
        offset_start = time.time()
    
    corrected_stack = stack.copy()  # Create a copy to avoid modifying the original
    
    if offset_correction == 'min_subtract':
        # For each z-plane, find minimum (or low percentile) value and subtract
        for z in range(z_max):
            # Use percentile instead of minimum to avoid outliers
            min_value = np.percentile(stack[:, z, :, :], percentile_value)
            corrected_stack[:, z, :, :] -= min_value
            if timing:
                print(f"Z-plane {z}: Subtracted {min_value:.2f} (percentile {percentile_value})")
    
    elif offset_correction == 'positive_shift':
        # Find global minimum and shift everything to make it positive
        global_min = np.min(stack)
        if global_min < 0:
            corrected_stack -= global_min - 1  # Add 1 to ensure positive values
            if timing:
                print(f"Applied global shift of {-(global_min - 1):.2f} to ensure positive values")
    
    elif offset_correction == 'manual' and global_offset is not None:
        # Add a manual global offset
        corrected_stack += global_offset
        if timing:
            print(f"Applied manual global offset of {global_offset:.2f}")
    
    elif offset_correction != 'none':
        print(f"Warning: Invalid offset_correction method '{offset_correction}'. No correction applied.")
    
    if timing:
        print(f"Offset correction: {time.time() - offset_start:.2f} seconds")
        baseline_start = time.time()
    
    # Initialize result array
    dff = np.zeros_like(corrected_stack)
    
    # Calculate ΔF/F with corrected stack
    if per_plane_baselines:
        # Process each z-plane with its own baseline frames
        for z in range(z_max):
            # Get baseline frames for this plane
            plane_frames = np.array(baseline_frames[z], dtype=int).flatten()
            
            # Calculate baseline for this plane
            f0_plane = np.mean(corrected_stack[plane_frames, z, :, :], axis=0)
            
            # Apply minimum threshold
            f0_plane = np.maximum(f0_plane, min_baseline_value)
            
            # Calculate ΔF/F for this plane
            dff[:, z, :, :] = (corrected_stack[:, z, :, :] - f0_plane) / f0_plane
    else:
        # Use the same baseline frames for all planes
        flat_frames = np.array(baseline_frames, dtype=int).flatten()
        
        # Calculate baseline for all planes
        f0 = np.mean(corrected_stack[flat_frames, :, :, :], axis=0)
        
        # Apply minimum threshold
        f0 = np.maximum(f0, min_baseline_value)
        
        # Calculate ΔF/F using vectorized operations
        dff = (corrected_stack - f0[np.newaxis, :, :, :]) / f0[np.newaxis, :, :, :]
    
    if timing:
        total_time = time.time() - start_time
        baseline_time = time.time() - baseline_start
        print(f"Baseline calculation: {baseline_time:.2f} seconds")
        print(f"Total ΔF/F calculation: {total_time:.2f} seconds")
    
    return dff


def analyze_data_statistics(stack, z_plane=0):
    """
    Analyze basic statistics of the data to help identify offset issues.
    
    Parameters
    ----------
    stack : np.ndarray
        4D array with dimensions (time, z, x, y)
    z_plane : int, optional
        Z-plane to analyze, by default 0
    """
    # Get a single z-plane
    plane_data = stack[:, z_plane, :, :]
    
    # Calculate basic statistics
    min_val = np.min(plane_data)
    max_val = np.max(plane_data)
    mean_val = np.mean(plane_data)
    p1 = np.percentile(plane_data, 1)
    p99 = np.percentile(plane_data, 99)
    
    print(f"Data statistics for z-plane {z_plane}:")
    print(f"  Min value: {min_val}")
    print(f"  Max value: {max_val}")
    print(f"  Mean value: {mean_val}")
    print(f"  1st percentile: {p1}")
    print(f"  99th percentile: {p99}")
    print(f"  Data range: {max_val - min_val}")
    
    # Check for potential offset issues
    if min_val < 0:
        print(f"  WARNING: Negative values detected. Min value: {min_val}")
        print(f"  Suggested offset correction: {-min_val + 1}")
    
    # Check for low baseline
    if min_val >= 0 and min_val < 10:
        print(f"  NOTE: Low minimum values detected ({min_val}). Consider adjusting min_baseline_value.")
    
    # Check for potential outliers
    if (max_val - p99) > (p99 - min_val) * 10:
        print(f"  WARNING: Potential high-value outliers. Consider using percentile-based methods.")

In [ ]:
master = Path(r"Z:\Hagar\e0075\v06\gad1b")
all_fish = list(master.glob("*_f*"))
print(all_fish)

In [ ]:
########Testing new dff function


for f in all_fish[-2:]:
    print(f)
    #try:
    if not (f / "dff3").exists():
        
        stack = SplitDataset(f / "aligned")[:,:,:,:]
        len_rec, num_planes, x_pix, y_pix = np.shape(stack)
        exp_list = glob(str(f / "behavior/*.json"))
        # make a list of sensory regressors for each plane
        plane_list = list(f.glob("suite2p\*00*"))
        reg_list = [None] * num_planes

        for i in range(len(plane_list)):
            new_reg = fl.load(plane_list[i] / 'sensory_regressors.h5')['regressors_conv']
            reg_list[i] = new_reg


        baseline_frames = [no_regressor_frames(reg) for reg in reg_list]
        
        analyze_data_statistics(stack, z_plane=0)
        
        dff_result = calculate_dff_with_offset_correction(
            stack, 
            baseline_frames,
            per_plane_baselines=True,
            offset_correction='min_subtract',  # Choose correction method
            percentile_value=1.0,             # Use 1st percentile for min_subtract
            timing=True                       # Show timing information
        )
        
        for i in range(len(plane_list)):
    
            if i < 10:
                file_name = '000' + str(i) + '.h5'
            else:
                file_name = '00' + str(i) + '.h5'

            data_out = dff_result[:,i,:,:]
            data_out = np.expand_dims(data_out, 1)

            dff_dir = (f / "dff2")
            dff_dir.mkdir(exist_ok=True)
            data_out_folder = str(f / "dff2" / file_name)
            fl.save(data_out_folder, {"stack_4D": data_out}, compression="blosc")
            
        shutil.copy(str(f / "aligned/stack_metadata.json"), str(f / "dff2"))